In [0]:
# Bronze Layer: Raw Shipments Ingestion
# Uses Auto Loader for incremental file ingestion
# Preserves raw data with added metadata columns

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp, input_file_name, lit,
    monotonically_increasing_id, col
)

spark = SparkSession.builder.getOrCreate()

# Configuration
LANDING_PATH  = "s3://ecom-landing-zone-2026/landing/shipments/"
CHECKPOINT    = "s3://ecom-landing-zone-2026/checkpoints/shipments_checkpoint/"
BRONZE_TABLE  = "ecomstore.ecomlake.bronze_shipments"
SOURCE_SYSTEM = "logistics_management_system"


spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
        shipment_id STRING, order_id STRING, carrier STRING, tracking_number STRING, 
        shipment_status STRING, dispatch_date STRING, promised_delivery_date STRING, 
        actual_delivery_date STRING, delivery_attempts STRING, warehouse_id STRING, 
        created_at STRING, updated_at STRING, _batch_id STRING, _rescued_data STRING,
        _ingestion_timestamp TIMESTAMP, _source_file_name STRING, 
        _pipeline_layer STRING, _source_system_override STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")

# Schema Hint 
# All columns are STRING at bronze. We don't cast here.
# Bronze preserves raw data. Silver does the casting.
shipments_schema = """
    shipment_id             STRING,
    order_id                STRING,
    carrier                 STRING,
    tracking_number         STRING,
    shipment_status         STRING,
    dispatch_date           STRING,
    promised_delivery_date  STRING,
    actual_delivery_date    STRING,
    delivery_attempts       STRING,
    warehouse_id            STRING,
    created_at              STRING,
    updated_at              STRING,
    _batch_id               STRING
"""

# Auto Loader Read
raw_shipments_df = (
    spark.readStream
    .format("cloudFiles")                               # Auto Loader format
    .option("cloudFiles.format", "csv")                 # source file format
    .option("cloudFiles.schemaLocation", CHECKPOINT + "schema/")
    .option("cloudFiles.inferColumnTypes", "false")     # keep everything as string
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # handle schema drift
    .option("header", "true")
    .option("multiLine", "false")
    .option("mode", "PERMISSIVE")                       # keep bad records with nulls
    .option("cloudFiles.schemaHints", shipments_schema)
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(LANDING_PATH)
    # Add ingestion metadata columns
    .withColumn("_ingestion_timestamp", current_timestamp())    # when did we ingest this?
    .withColumn("_source_file_name", col("_metadata.file_path"))         # which file did it come from?
    .withColumn("_pipeline_layer", lit("bronze"))
    .withColumn("_source_system_override", lit(SOURCE_SYSTEM))
)

# Write to Bronze Delta Table (Append Only)
(
    raw_shipments_df.writeStream
    .format("delta")
    .outputMode("append")                       # bronze is ALWAYS append-only
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")              # allow schema evolution
    .trigger(availableNow=True)                 # process all available files, then stop
    .toTable(BRONZE_TABLE)                      # writes to managed Delta table
)

print(f"✅ Bronze ingestion complete for: {BRONZE_TABLE}")